In [1]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import log_loss, brier_score_loss

# Find the project root by locating the src directory
current_path = Path.cwd()

project_root = next(
    path for path in [current_path, *current_path.parents]
    if (path / "src").is_dir()
)

# Make project modules importable
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

import src.feature_engineering as fe
from src.data_loader import load_cv_data

print("Project root:", project_root)

Project root: c:\Users\rajur\OneDrive\Desktop\next-move-ai


In [3]:
fold_datasets = load_cv_data(project_root)

# Convert skill IDs into Python sets
def convert_skill_id(skill_id):
    if pd.isna(skill_id):
        return set()

    return set(str(skill_id).split("_"))


for train_df, val_df in fold_datasets:

    train_df["skill_set"] = train_df["skill_id"].apply(
        convert_skill_id
    )

    val_df["skill_set"] = val_df["skill_id"].apply(
        convert_skill_id
    )


# Display fold sizes
for fold_number, (train_df, val_df) in enumerate(
    fold_datasets, start=1
):
    print(
        f"Fold {fold_number}: "
        f"Training = {train_df.shape}, "
        f"Validation = {val_df.shape}"
    )

Fold 1: Training = (177324, 7), Validation = (42659, 7)
Fold 2: Training = (176273, 7), Validation = (43710, 7)
Fold 3: Training = (170186, 7), Validation = (49797, 7)
Fold 4: Training = (178311, 7), Validation = (41672, 7)
Fold 5: Training = (177838, 7), Validation = (42145, 7)


In [4]:
v23_fold_datasets = []

for fold_number, (fold_train_df, fold_val_df) in enumerate(
    fold_datasets, start=1
):

    print(f"\nProcessing Fold {fold_number}...")

    # Generate training features
    v23_train_df = fe.add_same_skill_history_features(
        fold_train_df,
        alpha=10,
        prior_data=fold_train_df
    )

    # Generate validation features
    v23_val_df = fe.add_same_skill_history_features(
        fold_val_df,
        alpha=10,
        prior_data=fold_train_df
    )

    v23_fold_datasets.append(
        (v23_train_df, v23_val_df)
    )

    print("Training features:", v23_train_df.shape)
    print("Validation features:", v23_val_df.shape)

print("\nV2.3 base feature generation completed.")


Processing Fold 1...
Training features: (177324, 16)
Validation features: (42659, 16)

Processing Fold 2...
Training features: (176273, 16)
Validation features: (43710, 16)

Processing Fold 3...
Training features: (170186, 16)
Validation features: (49797, 16)

Processing Fold 4...
Training features: (178311, 16)
Validation features: (41672, 16)

Processing Fold 5...
Training features: (177838, 16)
Validation features: (42145, 16)

V2.3 base feature generation completed.


In [5]:
feature_columns_to_check = [
    "same_skill_previous_interactions",
    "same_skill_previous_correct",
    "same_skill_global_prior",
    "same_skill_smoothed_accuracy"
]

for fold_number, (train_df, val_df) in enumerate(
    v23_fold_datasets, start=1
):

    print(f"\n========== FOLD {fold_number} ==========")

    for name, df in [
        ("Training", train_df),
        ("Validation", val_df)
    ]:

        print(f"\n{name}:")

        print(
            "Missing values:",
            df[feature_columns_to_check].isna().sum().sum()
        )

        print(
            "Invalid historical counts:",
            (
                df["same_skill_previous_correct"]
                > df["same_skill_previous_interactions"]
            ).sum()
        )

        print(
            "Global prior range:",
            df["same_skill_global_prior"].min(),
            df["same_skill_global_prior"].max()
        )

        print(
            "Smoothed accuracy range:",
            df["same_skill_smoothed_accuracy"].min(),
            df["same_skill_smoothed_accuracy"].max()
        )


========== FOLD 1 ==========

Training:
Missing values: 0
Invalid historical counts: 0
Global prior range: 0.0 1.0
Smoothed accuracy range: 0.0 1.0

Validation:
Missing values: 0
Invalid historical counts: 0
Global prior range: 0.0 1.0
Smoothed accuracy range: 0.0 1.0

========== FOLD 2 ==========

Training:
Missing values: 0
Invalid historical counts: 0
Global prior range: 0.0 1.0
Smoothed accuracy range: 0.0 1.0

Validation:
Missing values: 0
Invalid historical counts: 0
Global prior range: 0.0 1.0
Smoothed accuracy range: 0.0 1.0

========== FOLD 3 ==========

Training:
Missing values: 0
Invalid historical counts: 0
Global prior range: 0.0 1.0
Smoothed accuracy range: 0.0 1.0

Validation:
Missing values: 0
Invalid historical counts: 0
Global prior range: 0.0 1.0
Smoothed accuracy range: 0.0 1.0

========== FOLD 4 ==========

Training:
Missing values: 0
Invalid historical counts: 0
Global prior range: 0.0 1.0
Smoothed accuracy range: 0.0 1.0

Validation:
Missing values: 0
Invalid hi

In [6]:
for train_df, val_df in v23_fold_datasets:

    train_df["log_same_skill_previous_interactions"] = np.log1p(
        train_df["same_skill_previous_interactions"]
    )

    val_df["log_same_skill_previous_interactions"] = np.log1p(
        val_df["same_skill_previous_interactions"]
    )

print("Log-experience feature created for all folds.")

Log-experience feature created for all folds.


In [7]:
for fold_number, (train_df, val_df) in enumerate(
    v23_fold_datasets, start=1
):

    print(f"\n========== FOLD {fold_number} ==========")

    for name, df in [
        ("Training", train_df),
        ("Validation", val_df)
    ]:

        print(f"\n{name}:")

        print(
            "Missing log-experience values:",
            df["log_same_skill_previous_interactions"].isna().sum()
        )

        print(
            "Minimum log experience:",
            df["log_same_skill_previous_interactions"].min()
        )

        print(
            "Maximum log experience:",
            df["log_same_skill_previous_interactions"].max()
        )


========== FOLD 1 ==========

Training:
Missing log-experience values: 0
Minimum log experience: 0.0
Maximum log experience: 5.081404364984463

Validation:
Missing log-experience values: 0
Minimum log experience: 0.0
Maximum log experience: 4.859812404361672

========== FOLD 2 ==========

Training:
Missing log-experience values: 0
Minimum log experience: 0.0
Maximum log experience: 4.990432586778736

Validation:
Missing log-experience values: 0
Minimum log experience: 0.0
Maximum log experience: 5.081404364984463

========== FOLD 3 ==========

Training:
Missing log-experience values: 0
Minimum log experience: 0.0
Maximum log experience: 5.081404364984463

Validation:
Missing log-experience values: 0
Minimum log experience: 0.0
Maximum log experience: 4.969813299576001

========== FOLD 4 ==========

Training:
Missing log-experience values: 0
Minimum log experience: 0.0
Maximum log experience: 5.081404364984463

Validation:
Missing log-experience values: 0
Minimum log experience: 0.0
Ma

In [8]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

from sklearn.metrics import (
    log_loss,
    brier_score_loss,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)

# Two input features for V2.3
feature_columns = [
    "same_skill_smoothed_accuracy",
    "log_same_skill_previous_interactions"
]

v23_models = []
v23_results = []

for fold_number, (train_data, val_data) in enumerate(
    v23_fold_datasets, start=1
):

    X_train = train_data[feature_columns]
    y_train = train_data["correct"]

    X_val = val_data[feature_columns]
    y_val = val_data["correct"]

    # Same preprocessing and classifier as V2.1
    model = Pipeline([
        ("scaler", StandardScaler()),
        ("classifier", LogisticRegression(max_iter=1000))
    ])

    # Train using training data only
    model.fit(X_train, y_train)

    # Predict probabilities and class labels
    y_pred_proba = model.predict_proba(X_val)[:, 1]
    y_pred = model.predict(X_val)

    # Probability-based evaluation
    ll = log_loss(y_val, y_pred_proba, labels=[0, 1])
    bs = brier_score_loss(y_val, y_pred_proba)

    # Classification-based evaluation (default threshold = 0.5)
    accuracy = accuracy_score(y_val, y_pred)
    precision = precision_score(y_val, y_pred, zero_division=0)
    recall = recall_score(y_val, y_pred, zero_division=0)
    f1 = f1_score(y_val, y_pred, zero_division=0)
    macro_f1 = f1_score(y_val, y_pred, average="macro", zero_division=0)

    v23_models.append(model)

    v23_results.append({
        "fold": fold_number,
        "log_loss": ll,
        "brier_score": bs,
        "accuracy": accuracy,
        "precision": precision,
        "recall": recall,
        "f1_score": f1,
        "macro_f1": macro_f1
    })

    print(f"\nFold {fold_number}")
    print(f"Log Loss:  {ll:.4f}")
    print(f"Brier:     {bs:.4f}")
    print(f"Accuracy:  {accuracy:.4f}")
    print(f"Precision: {precision:.4f}")
    print(f"Recall:    {recall:.4f}")
    print(f"F1-score:  {f1:.4f}")
    print(f"Macro F1:  {macro_f1:.4f}")


Fold 1
Log Loss:  0.5916
Brier:     0.2026
Accuracy:  0.6955
Precision: 0.7025
Recall:    0.9251
F1-score:  0.7986
Macro F1:  0.5877

Fold 2
Log Loss:  0.5853
Brier:     0.1998
Accuracy:  0.7043
Precision: 0.7182
Recall:    0.9250
F1-score:  0.8086
Macro F1:  0.5794

Fold 3
Log Loss:  0.5985
Brier:     0.2058
Accuracy:  0.6843
Precision: 0.6900
Recall:    0.9227
F1-score:  0.7896
Macro F1:  0.5791

Fold 4
Log Loss:  0.5911
Brier:     0.2023
Accuracy:  0.6966
Precision: 0.7055
Recall:    0.9310
F1-score:  0.8027
Macro F1:  0.5729

Fold 5
Log Loss:  0.5865
Brier:     0.2004
Accuracy:  0.7003
Precision: 0.7094
Recall:    0.9286
F1-score:  0.8043
Macro F1:  0.5825


In [9]:
v23_results_df = pd.DataFrame(v23_results)

print("V2.3 Cross-Validation Results")
display(v23_results_df.round(4))

print("\nAverage Performance:")

metrics = [
    "log_loss",
    "brier_score",
    "accuracy",
    "precision",
    "recall",
    "f1_score",
    "macro_f1"
]

for metric in metrics:
    print(
        f"Average {metric}: "
        f"{v23_results_df[metric].mean():.4f}"
    )

V2.3 Cross-Validation Results


,fold,log_loss,brier_score,accuracy,precision,recall,f1_score,macro_f1
0,1,0.5916,0.2026,0.6955,0.7025,0.9251,0.7986,0.5877
1,2,0.5853,0.1998,0.7043,0.7182,0.9250,0.8086,0.5794
2,3,0.5985,0.2058,0.6843,0.6900,0.9227,0.7896,0.5791
3,4,0.5911,0.2023,0.6966,0.7055,0.9310,0.8027,0.5729
4,5,0.5865,0.2004,0.7003,0.7094,0.9286,0.8043,0.5825



Average Performance:
Average log_loss: 0.5906
Average brier_score: 0.2022
Average accuracy: 0.6962
Average precision: 0.7051
Average recall: 0.9265
Average f1_score: 0.8008
Average macro_f1: 0.5803


In [10]:
# Collect out-of-fold predictions for V2.3

v23_predictions = []

for fold_number, (train_data, val_data) in enumerate(
    v23_fold_datasets, start=1
):

    # Retrieve the already-trained model for this fold
    model = v23_models[fold_number - 1]

    # Generate validation probabilities
    X_val = val_data[feature_columns]
    y_pred_proba = model.predict_proba(X_val)[:, 1]

    # Store actual outcomes and predictions
    fold_predictions = pd.DataFrame({
        "user_id": val_data["user_id"].values,
        "order_id": val_data["order_id"].values,
        "fold": fold_number,
        "y_true": val_data["correct"].values,
        "y_pred_proba": y_pred_proba,
        "model": "V2.3"
    })

    v23_predictions.append(fold_predictions)

# Combine predictions from all five folds
v23_predictions_df = pd.concat(
    v23_predictions,
    ignore_index=True
)

print("Prediction collection completed.")
print("Total predictions:", len(v23_predictions_df))
display(v23_predictions_df.head())

Prediction collection completed.
Total predictions: 219983


,user_id,order_id,fold,y_true,y_pred_proba,model
0,21825,21616385,1,0,0.436079,V2.3
1,21825,21616478,1,1,0.399426,V2.3
2,21825,21616677,1,1,0.462016,V2.3
3,21825,21616919,1,1,0.515339,V2.3
4,21825,21671620,1,0,0.601361,V2.3


In [11]:
print("Total rows:", len(v23_predictions_df))

print("\nMissing values:")
print(v23_predictions_df.isna().sum())

print("\nFold distribution:")
print(v23_predictions_df["fold"].value_counts().sort_index())

print("\nProbability range:")
print(
    v23_predictions_df["y_pred_proba"].min(),
    v23_predictions_df["y_pred_proba"].max()
)

print("\nActual outcome distribution:")
print(v23_predictions_df["y_true"].value_counts())

Total rows: 219983

Missing values:
user_id         0
order_id        0
fold            0
y_true          0
y_pred_proba    0
model           0
dtype: int64

Fold distribution:
fold
1    42659
2    43710
3    49797
4    41672
5    42145
Name: count, dtype: int64

Probability range:
0.08695443900025811 0.9242385359218169

Actual outcome distribution:
y_true
1    144886
0     75097
Name: count, dtype: int64


In [12]:
prediction_dir = (
    project_root / "data" / "processed" / "model_predictions"
)

prediction_dir.mkdir(parents=True, exist_ok=True)

v23_prediction_path = prediction_dir / "v2_3_predictions.csv"

v23_predictions_df.to_csv(
    v23_prediction_path,
    index=False
)

print("V2.3 predictions saved successfully.")
print("File:", v23_prediction_path)

V2.3 predictions saved successfully.
File: c:\Users\rajur\OneDrive\Desktop\next-move-ai\data\processed\model_predictions\v2_3_predictions.csv


In [13]:
feature_dir = (
    project_root / "data" / "processed" / "engineered_features"
    / "group_2_same_skill_history" / "v2.3"
)
feature_dir.mkdir(parents=True, exist_ok=True)

columns_to_save = [
    "user_id",
    "order_id",
    "correct",
    "same_skill_smoothed_accuracy",
    "log_same_skill_previous_interactions"
]

for fold_number, (train_df, val_df) in enumerate(v23_fold_datasets, start=1):
    train_df[columns_to_save].to_csv(
        feature_dir / f"fold_{fold_number}_train.csv", index=False
    )
    val_df[columns_to_save].to_csv(
        feature_dir / f"fold_{fold_number}_validation.csv", index=False
    )
    print(f"Fold {fold_number} saved.")

print("V2.3 features saved successfully!")

Fold 1 saved.
Fold 2 saved.
Fold 3 saved.
Fold 4 saved.
Fold 5 saved.
V2.3 features saved successfully!
